In [ ]:
# # volcano_plots.py
# # ----------------------------------------------------------
# # 1. Imports  (install once:  pip install pandas numpy scipy statsmodels adjustText matplotlib)
# import pandas as pd
# import numpy as np
# from scipy.stats import fisher_exact
# from statsmodels.stats.multitest import multipletests
# import matplotlib.pyplot as plt
# from adjustText import adjust_text
# from pathlib import Path

# # ----------------------------------------------------------
# # 2. User settings
# CSV_PATH       = "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_results_bg_mean_16hpf_20250611/enhanced_comprehensive_summary/motif_occurrence_table_averaged_normalized.csv"
# #"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_results_max_bg_all_20250610/comprehensive_summary/motif_occurrence_table_combined_summary.csv"
# #"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_results_bg_max_16hpf_20250610_combined/comprehensive_summary/motif_occurrence_table_combined_summary.csv"
# #"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_results_bg_mean_16hpf_20250611/comprehensive_summary/motif_occurrence_table_combined_summary.csv"       # << path to the file you supplied
# OUTDIR         = Path("volcano_plots_bg_mean_16hpf_20250611_enhanced")     # outputs will be saved here
# PSEUDOCOUNT    = 0.5                       # avoids log2(0)
# ALPHA          = 0.05                      # p-value threshold
# LABEL_TOP_N    = 10                        # how many motifs to label per plot

# OUTDIR.mkdir(exist_ok=True)

# # ----------------------------------------------------------
# # 3. Load & tidy
# df = pd.read_csv(CSV_PATH)

# # Your sheet has a meta-row called “Total” – drop it so only real motifs remain
# motifs_df = df[df["motif"] != "Total"].copy()

# # Pick the biological columns (all except motif name + overall Total)
# cell_types = [c for c in motifs_df.columns if c not in ("motif", "Total")]

# # Pre-compute column totals once (total motifs observed in each cell type)
# col_totals = motifs_df[cell_types].sum()

# # Global total (all cell types pooled)
# grand_total = col_totals.sum()

# # ----------------------------------------------------------
# # 4. Per-cell-type analysis + plotting
# for ct in cell_types:
#     # a) 2 × 2 contingency counts for every motif
#     in_ct       = motifs_df[ct].to_numpy()
#     in_rest     = motifs_df[cell_types].sum(axis=1).to_numpy() - in_ct
#     other_in_ct = col_totals[ct] - in_ct
#     other_rest  = (grand_total - col_totals[ct]) - in_rest

#     # b) Fisher exact test (one-sided enrichment)
#     pvals = np.array([fisher_exact([[a, b], [c, d]], alternative="greater")[1]
#                       for a, b, c, d in zip(in_ct, in_rest, other_in_ct, other_rest)])

#     # c) Optional multiple-test correction (kept here for reference)
#     #    If you prefer adjusted q-values, replace `pvals` with `qvals` below.
#     # _, qvals, _, _ = multipletests(pvals, method="fdr_bh")

#     # d) log₂ fold change of motif frequency (add pseudocount to both numerators)
#     prop_in_ct   = (in_ct   + PSEUDOCOUNT) / col_totals[ct]
#     prop_in_rest = (in_rest + PSEUDOCOUNT) / (grand_total - col_totals[ct])
#     log2fc = np.log2(prop_in_ct / prop_in_rest)

#     # e) Significance mask for colouring
#     sig_mask = (pvals < ALPHA) & (log2fc > 0)

#     # f) Volcano plot
#     plt.figure(figsize=(5, 5))
#     plt.scatter(log2fc, -np.log10(pvals),
#                 s=20,
#                 c=np.where(sig_mask, "darkorange", "steelblue"),
#                 alpha=0.85)
#     plt.axhline(-np.log10(ALPHA), ls="--", lw=0.8, color="grey")
#     plt.axvline(0,              ls="--", lw=0.8, color="grey")
#     plt.xlabel("log₂ fold-change (vs. rest)")
#     plt.ylabel("–log₁₀ p-value (Fisher exact)")
#     plt.title(f"Motif enrichment in {ct}")

#     # g) Label the top N enriched motifs
#     top_idx = np.argsort(pvals[sig_mask])[:LABEL_TOP_N]          # smallest p-values
#     texts   = []
#     motif_names = motifs_df["motif"].to_numpy()
#     for idx in np.where(sig_mask)[0][top_idx]:
#         texts.append(
#             plt.text(log2fc[idx], -np.log10(pvals[idx]),
#                      motif_names[idx], fontsize=7, ha="left", va="center")
#         )
#     adjust_text(texts, arrowprops={"arrowstyle": "-", "lw": 0.3})

#     # h) Save figure
#     safe_name = ct.replace(" ", "_").replace("/", "_")
#     plt.tight_layout()
#     plt.savefig(OUTDIR / f"volcano_{safe_name}.png", dpi=300)
#     plt.close()

# print(f"Done! {len(cell_types)} volcano plots written to {OUTDIR.resolve()}")


In [ ]:
# volcano_plots_zscore.py
# ----------------------------------------------------------
# Z-score based approach for averaged normalized data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text
from pathlib import Path

# ----------------------------------------------------------
# User settings
CSV_PATH = "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_results_bg_mean_16hpf_20250612/enhanced_comprehensive_summary/motif_occurrence_table_averaged_normalized.csv"
OUTDIR = Path("volcano_plots_normalized_perc75_20250612")
FOLD_CHANGE_THRESHOLD = 1.2  # Lower threshold for more sensitivity
ZSCORE_THRESHOLD = 1.0       # Lower threshold for more sensitivity
LABEL_TOP_N = 10

OUTDIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load & tidy
df = pd.read_csv(CSV_PATH)
motifs_df = df[df["motif"] != "Total"].copy()
cell_types = [c for c in motifs_df.columns if c not in ("motif", "Total")]

# ----------------------------------------------------------
# Per-cell-type analysis + plotting
for ct in cell_types:
    print(f"Processing {ct}...")
    
    # Get data for this analysis
    data_matrix = motifs_df[cell_types].values  # All cell type values
    target_col_idx = cell_types.index(ct)
    
    # Calculate statistics for each motif
    results = []
    for i, motif_name in enumerate(motifs_df["motif"]):
        motif_values = data_matrix[i, :]  # This motif across all cell types
        target_value = motif_values[target_col_idx]
        other_values = np.concatenate([motif_values[:target_col_idx], 
                                     motif_values[target_col_idx+1:]])
        
        # Method 1: Z-score approach
        if len(other_values) > 1 and np.std(other_values) > 0:
            zscore = (target_value - np.mean(other_values)) / np.std(other_values)
        else:
            zscore = 0
        
        # Method 2: Fold-change approach
        mean_others = np.mean(other_values)
        if mean_others > 0:
            fold_change = target_value / mean_others
            log2fc = np.log2(fold_change)
        else:
            fold_change = 1
            log2fc = 0
        
        # Method 3: Percentile rank approach
        all_values_sorted = np.sort(motif_values)
        percentile_rank = (np.searchsorted(all_values_sorted, target_value) / len(all_values_sorted)) * 100
        
        results.append({
            'motif': motif_name,
            'target_value': target_value,
            'mean_others': mean_others,
            'zscore': zscore,
            'log2fc': log2fc,
            'fold_change': fold_change,
            'percentile_rank': percentile_rank
        })
    
    results_df = pd.DataFrame(results)
    
    # Define significance based on multiple criteria
    sig_mask = (
        (np.abs(results_df['zscore']) >= ZSCORE_THRESHOLD) & 
        (results_df['fold_change'] >= FOLD_CHANGE_THRESHOLD) &
        (results_df['log2fc'] > 0)  # Only enriched, not depleted
    )
    
    print(f"  Found {sig_mask.sum()} significant motifs out of {len(results_df)}")
    
    # Create "pseudo p-values" from z-scores for visualization
    # Convert z-scores to pseudo p-values (higher z-score = lower p-value)
    from scipy.stats import norm
    pseudo_pvals = 2 * (1 - norm.cdf(np.abs(results_df['zscore'])))  # Two-tailed
    pseudo_pvals = np.clip(pseudo_pvals, 1e-10, 1)  # Avoid log(0)
    
    # Volcano plot - MADE SQUARE
    plt.figure(figsize=(8, 8))  # Changed from (12, 8) to square
    
    # Plot points - MADE BIGGER
    scatter = plt.scatter(results_df['log2fc'], -np.log10(pseudo_pvals),
                         s=60,  # Increased from 40
                         c=np.where(sig_mask, "darkorange", "steelblue"),
                         alpha=0.7,
                         edgecolors='black',
                         linewidths=0.5)
    
    # Add threshold lines
    plt.axhline(-np.log10(2 * (1 - norm.cdf(ZSCORE_THRESHOLD))), 
                ls="--", lw=1.5, color="red", alpha=0.7,  # Thicker line
                label=f'Z-score = ±{ZSCORE_THRESHOLD}')
    plt.axvline(np.log2(FOLD_CHANGE_THRESHOLD), 
                ls="--", lw=1.5, color="green", alpha=0.7,  # Thicker line
                label=f'FC = {FOLD_CHANGE_THRESHOLD}')
    plt.axvline(0, ls="--", lw=1, color="grey", alpha=0.5)
    
    # BIGGER AXIS LABELS
    plt.xlabel(f"log₂ fold-change ({ct} vs. other cell types)", fontsize=14, fontweight='bold')
    plt.ylabel("–log₁₀ pseudo p-value (from Z-score)", fontsize=14, fontweight='bold')
    plt.title(f"Motif enrichment in {ct}\n(Z-score and fold-change based)", 
              fontsize=16, fontweight='bold', pad=20)
    
    # BIGGER TICK LABELS
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    # Add statistics to plot - BIGGER TEXT
    stats_text = f"""Significant: {sig_mask.sum()}
Total: {len(results_df)}
Criteria:
• Z-score ≥ {ZSCORE_THRESHOLD}
• Fold-change ≥ {FOLD_CHANGE_THRESHOLD}
• log₂FC > 0"""
    
    plt.text(0.02, 0.98, stats_text, 
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
             fontsize=11)  # Increased from 9
    
    # Get top motifs by two criteria
    # 1. Top 10 most significant (highest z-score)
    top_by_zscore = results_df.nlargest(LABEL_TOP_N, 'zscore').index
    
    # 2. Top 10 highest fold-change (but only if log2fc > 0)
    enriched_motifs = results_df[results_df['log2fc'] > 0]
    if len(enriched_motifs) >= LABEL_TOP_N:
        top_by_fc = enriched_motifs.nlargest(LABEL_TOP_N, 'log2fc').index
    else:
        top_by_fc = enriched_motifs.index
    
    # Combine and remove duplicates
    top_indices = list(set(top_by_zscore) | set(top_by_fc))
    
    print(f"  Labeling {len(top_indices)} motifs (top by significance + fold-change)")
    
    # Create labels with different colors/styles for different categories - BIGGER LABELS
    texts = []
    for idx in top_indices:
        motif_name = results_df.loc[idx, 'motif']
        x_pos = results_df.loc[idx, 'log2fc']
        y_pos = -np.log10(pseudo_pvals[idx])
        
        # Determine label style based on category
        if idx in top_by_zscore and idx in top_by_fc:
            # In both top lists - use bold red
            label_color = 'darkred'
            fontweight = 'bold'
            label_prefix = '★'  # Star for top in both categories
        elif idx in top_by_zscore:
            # Top by significance - use red
            label_color = 'red'
            fontweight = 'normal'
            label_prefix = 'S'  # S for significance
        else:
            # Top by fold-change - use blue
            label_color = 'blue'
            fontweight = 'normal'
            label_prefix = 'F'  # F for fold-change
        
        texts.append(
            plt.text(x_pos, y_pos, f"{label_prefix}{motif_name}", 
                   fontsize=10, ha="left", va="bottom",  # Increased from 8
                   color=label_color, fontweight=fontweight,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white',  # More padding
                           alpha=0.9, edgecolor=label_color, linewidth=0.8))  # Thicker border
        )
    
    # Apply adjustText to avoid overlaps
    if texts:
        adjust_text(texts, 
                   arrowprops={"arrowstyle": "->", "lw": 0.8, "alpha": 0.8},  # Thicker arrows
                   avoid_points=True,
                   expand_text=(1.2, 1.3),  # More space
                   expand_points=(1.2, 1.3))
    
    # Add custom legend for label categories - BIGGER LEGEND
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='darkred', label='★ Top in both significance & fold-change'),
        Patch(facecolor='red', label='S Top by significance (Z-score)'),
        Patch(facecolor='blue', label='F Top by fold-change'),
        plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.7, 
                  label=f'Z-score = ±{ZSCORE_THRESHOLD}'),
        plt.Line2D([0], [0], color='green', linestyle='--', alpha=0.7, 
                  label=f'FC = {FOLD_CHANGE_THRESHOLD}')
    ]
    
    # Position legend to avoid blocking data points - BIGGER LEGEND
    plt.legend(handles=legend_elements, 
              loc='lower right',
              fontsize=10,  # Increased from 8
              framealpha=0.95,
              fancybox=True,
              shadow=True)
    
    plt.grid(True, alpha=0.3, linewidth=0.8)  # Slightly thicker grid
    plt.tight_layout(pad=1.5)  # More padding
    
    # Save figure with higher DPI for better quality
    safe_name = ct.replace(" ", "_").replace("/", "_")
    plt.savefig(OUTDIR / f"volcano_{safe_name}.png", dpi=400, bbox_inches='tight',  # Higher DPI
                facecolor='white', edgecolor='none')
    plt.close()
    
    # Save detailed results for this cell type with labeling info
    results_df['significant'] = sig_mask
    results_df['top_by_zscore'] = results_df.index.isin(top_by_zscore)
    results_df['top_by_fc'] = results_df.index.isin(top_by_fc)
    results_df['labeled'] = results_df.index.isin(top_indices)
    results_df.to_csv(OUTDIR / f"enrichment_results_{safe_name}.csv", index=False)

print(f"Done! {len(cell_types)} volcano plots written to {OUTDIR.resolve()}")